# Cellpose-SAM segmentation in Google Colab

Draws one outline (ROI) per cell on the **brightfield image** and saves them as an ImageJ ROI zip,
for the calcium imaging pipeline (see the repository README).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Upload your image to Google Drive,
e.g. into a folder `cellpose`. Run the cells from top to bottom.

The segmentation code here is **identical** to `scripts/segment_cells.py`, and the Cellpose version is
pinned to the same version as in the repository, so Colab and your own computer give the same ROIs.
Check it on your image with `scripts/compare_rois.py`.


## 1 · Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Install Cellpose (1–2 min)
The version is pinned so the result matches the terminal script.

In [ ]:
!pip install -q cellpose==4.2.1.1 roifile tifffile

## 3 · Settings and segmentation
Change only the four settings and `IMAGE_PATH`. First run with `DIAMETER = 0`.

| If you see | Change |
|---|---|
| cells missed | `CELLPROB_THRESHOLD = -1` or `FLOW_THRESHOLD = 0.6` |
| debris or background outlined | `CELLPROB_THRESHOLD = 1` |
| one side of the image darker | `FLATTEN_ILLUMINATION = True` |
| cells split in pieces or merged | `DIAMETER =` the median diameter printed below |


In [ ]:
# ================= SETTINGS =================
IMAGE_PATH = "/content/drive/MyDrive/cellpose/brightfield.tif"
DIAMETER = 0                  # 0 = no resizing (Cellpose-SAM default); else the cell diameter in pixels
CELLPROB_THRESHOLD = 0.0      # lower (e.g. -1) = more cells, higher (e.g. 1) = fewer
FLOW_THRESHOLD = 0.4          # higher (e.g. 0.6) = accepts less perfect shapes
FLATTEN_ILLUMINATION = False  # True if one side of the image is brighter than the other
# ============================================

import numpy as np

# ---------- the segmentation itself: IDENTICAL to the code in scripts/segment_cells.py ----------
def load_image(path):
    """Read the image as one 2D grey image (RGB -> grey, several frames -> average)."""
    from cellpose import io
    img = io.imread(str(path)).astype(np.float32)
    print("Image shape as loaded:", img.shape)
    if img.ndim == 3:
        if img.shape[-1] in (3, 4):          # RGB -> gray
            img = img[..., :3].mean(axis=-1)
        else:                                # several frames -> average
            img = img.mean(axis=0)
    print(f"Image used: {img.shape[1]} x {img.shape[0]} pixels (width x height) -> must match the movies")
    return img


def segment(img, diameter=0, cellprob_threshold=0.0, flow_threshold=0.4, flatten_illumination=False):
    """Run Cellpose-SAM. Returns the label image (0 = background, 1..N = cells)."""
    from cellpose import models
    from scipy.ndimage import gaussian_filter
    if flatten_illumination:
        img = img / (gaussian_filter(img, sigma=100) + 1e-6)
    model = models.CellposeModel(gpu=True)    # uses the GPU if there is one, otherwise the CPU
    masks, flows, styles = model.eval(
        img,
        diameter=None if diameter == 0 else diameter,
        cellprob_threshold=cellprob_threshold,
        flow_threshold=flow_threshold,
    )
    return masks

img = load_image(IMAGE_PATH)
masks = segment(img, DIAMETER, CELLPROB_THRESHOLD, FLOW_THRESHOLD, FLATTEN_ILLUMINATION)

from cellpose import utils
import matplotlib.pyplot as plt
n_cells = int(masks.max())
print(f"Cells found: {n_cells}")
print(f"Median diameter: {utils.diameters(masks)[0]:.1f} px")
lo, hi = np.percentile(img, (1, 99))
fig, ax = plt.subplots(1, 2, figsize=(20, 9))
ax[0].imshow(img, cmap="gray", vmin=lo, vmax=hi); ax[0].set_title("Image")
ax[1].imshow(img, cmap="gray", vmin=lo, vmax=hi)
for o in utils.outlines_list(masks):
    ax[1].plot(o[:, 0], o[:, 1], color="yellow", linewidth=0.8)
ax[1].set_title(f"{n_cells} cells")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 4 · Save and download
Run only when the outlines look good. The files are also saved next to the image on your Drive.

In [ ]:
import os, json, tifffile
from cellpose import io
from google.colab import files
from importlib.metadata import version

stem = os.path.splitext(IMAGE_PATH)[0]
io.save_rois(masks, stem)                                  # -> <name>_rois.zip
tifffile.imwrite(stem + "_masks.tif", masks.astype(np.uint16))
fig.savefig(stem + "_outlines.png", dpi=150, bbox_inches="tight")
info = dict(diameter=DIAMETER, cellprob_threshold=CELLPROB_THRESHOLD, flow_threshold=FLOW_THRESHOLD,
            flatten_illumination=FLATTEN_ILLUMINATION, image=os.path.basename(IMAGE_PATH),
            image_size_px=[int(img.shape[1]), int(img.shape[0])], cells=int(masks.max()),
            median_diameter_px=round(float(utils.diameters(masks)[0]), 2), cellpose_version=version("cellpose"))
open(stem + "_segmentation.json", "w").write(json.dumps(info, indent=2))
print("Saved on Drive:", stem + "_rois.zip", "| cells:", masks.max())
files.download(stem + "_rois.zip")